# Finding How the Data Is Distributed

This is my own practice notebook based on what I learned in the IBM Data Analyst Capstone Project on Coursera.

## Goal

Practise checking how values are distributed across a survey dataset using counts, charts, language comparisons, correlation, crosstab analysis, and CSV export.

## Workflow rule

- `df` = original loaded dataset
- `df_cleaned` = cleaned working dataset
- `corr_df` = separate dataframe only for correlation analysis


## Step 1: Import libraries and load data


In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Load the dataset
data_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(data_url)

# Preview the first rows
df.head()


## Step 2: Examine the structure of the data


In [ ]:
# Show column names
print(df.columns)

# Show data types
print(df.dtypes)

# Show dataframe structure and non-null counts
df.info()


## Step 3: Handle missing data


In [ ]:
# Check for missing values in each column
print(df.isnull().sum())

# Create a cleaned working copy of the dataset
df_cleaned = df.copy()

# Fill selected missing values with clear labels
df_cleaned['RemoteWork'] = df_cleaned['RemoteWork'].fillna('Unknown')
df_cleaned['JobSat'] = df_cleaned['JobSat'].fillna('Not answered')

# Verify missing values in the cleaned columns
print(df_cleaned[['RemoteWork', 'JobSat']].isnull().sum())


## Step 4: Analyze key columns


In [ ]:
# Count each Employment category
print(df_cleaned['Employment'].value_counts())

# Count each JobSat value
print(df_cleaned['JobSat'].value_counts())

# Count each YearsCodePro value
print(df_cleaned['YearsCodePro'].value_counts())


## Step 5: Visualize Job Satisfaction


In [ ]:
# Keep only real JobSat answers and convert them to numbers
job_sat = df_cleaned[df_cleaned['JobSat'] != 'Not answered']['JobSat'].astype(float)

# Create a KDE plot for JobSat distribution
sns.kdeplot(job_sat, fill=True)

plt.title('Distribution of Job Satisfaction')
plt.xlabel('Job Satisfaction')
plt.ylabel('Density')
plt.show()

# Count JobSat values, excluding "Not answered" if it exists
job_counts = df_cleaned[df_cleaned['JobSat'] != 'Not answered']['JobSat'].value_counts()

# Create a pie chart for JobSat distribution
plt.pie(job_counts, labels=job_counts.index, autopct='%1.1f%%', startangle=90)

plt.title('Job Satisfaction Distribution')
plt.show()


## Step 6: Programming Languages Analysis


In [ ]:
# Split and count languages respondents have worked with
have_langs = df_cleaned['LanguageHaveWorkedWith'].dropna().str.split(';').sum()
have_counts = Counter(have_langs)

# Split and count languages respondents want to work with
want_langs = df_cleaned['LanguageWantToWorkWith'].dropna().str.split(';').sum()
want_counts = Counter(want_langs)

# Create a dataframe to compare used vs wanted languages
df_langs = pd.DataFrame({
    'HaveWorkedWith': pd.Series(have_counts),
    'WantToWorkWith': pd.Series(want_counts)
}).fillna(0)

# Select the top 10 languages based on current usage
top_langs = df_langs.sort_values('HaveWorkedWith', ascending=False).head(10)

# Plot the top languages
top_langs.plot(kind='bar', figsize=(10, 6))

plt.title('Top Programming Languages: Used vs Wanted')
plt.xlabel('Programming Language')
plt.ylabel('Number of Respondents')
plt.xticks(rotation=45, ha='right')
plt.show()


## Step 7: Analyze Remote Work Trends


In [ ]:
# Count each RemoteWork category in the cleaned dataset
remote_counts = df_cleaned['RemoteWork'].value_counts()

# Show the counts
print(remote_counts)

# Plot RemoteWork distribution
remote_counts.plot(kind='bar', figsize=(8, 5))

plt.title('Remote Work Distribution')
plt.xlabel('Remote Work Type')
plt.ylabel('Number of Respondents')
plt.xticks(rotation=45, ha='right')
plt.show()


## Step 8: Correlation between Job Satisfaction and Experience


In [ ]:
# Create a separate dataframe only for correlation analysis
corr_df = df_cleaned[['JobSat', 'YearsCodePro']].copy()

# Keep only real JobSat answers
corr_df = corr_df[corr_df['JobSat'] != 'Not answered']

# Replace special text values in YearsCodePro
corr_df['YearsCodePro'] = corr_df['YearsCodePro'].replace({
    'Less than 1 year': 0,
    'More than 50 years': 51
})

# Convert JobSat and YearsCodePro to numbers
corr_df['JobSat'] = pd.to_numeric(corr_df['JobSat'], errors='coerce')
corr_df['YearsCodePro'] = pd.to_numeric(corr_df['YearsCodePro'], errors='coerce')

# Remove rows with missing JobSat or YearsCodePro
corr_df = corr_df[['JobSat', 'YearsCodePro']].dropna()

# Calculate Pearson correlation
pearson_corr = corr_df['JobSat'].corr(corr_df['YearsCodePro'], method='pearson')

print("Pearson correlation between JobSat and YearsCodePro:")
print(pearson_corr)


## Step 9: Cross-tabulation Analysis: Employment vs Education Level


In [ ]:
# Create a full cross-tabulation table between education level and employment type
employment_edu = pd.crosstab(
    df_cleaned['EdLevel'],
    df_cleaned['Employment']
)

# Show the full table with all employment categories
employment_edu


In [ ]:
# Find the top 5 most common employment types
top_employment_types = df_cleaned['Employment'].value_counts().head(5).index

# Keep only rows that belong to the top 5 employment types
employment_edu_top = df_cleaned[df_cleaned['Employment'].isin(top_employment_types)]

# Create a smaller cross-tabulation table for the chart
employment_edu_chart = pd.crosstab(
    employment_edu_top['EdLevel'],
    employment_edu_top['Employment']
)

# Create a readable stacked bar chart
employment_edu_chart.plot(kind='bar', stacked=True, figsize=(12, 6))

# Add chart title and labels
plt.title('Employment Type by Education Level')
plt.xlabel('Education Level')
plt.ylabel('Number of Respondents')

# Rotate x-axis labels so long education labels are easier to read
plt.xticks(rotation=45, ha='right')

# Move the legend outside the chart
plt.legend(title='Employment Type', bbox_to_anchor=(1.05, 1), loc='upper left')

# Adjust layout so labels and legend fit better
plt.tight_layout()

# Show the chart
plt.show()


## Step 10: Export Cleaned Data


In [ ]:
# Save the cleaned dataframe as a CSV file
df_cleaned.to_csv('cleaned_survey_data.csv', index=False)

# Confirm that the file was saved
print("Cleaned dataset saved successfully as cleaned_survey_data.csv")


## Final note

This notebook keeps the workflow consistent by using `df_cleaned` after the missing-value handling step and by using `corr_df` only for correlation analysis.
